In [70]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [160]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [161]:
from src.simulations.benchmarks.hf import hf
from src.simulations.molecules import MoleculeSimulator

In [177]:
from src.utils.molecule_utils import compute_cc
from src.utils.procrustes_utils import (
    compute_procrustes_matrices,
    compute_cc_with_procrustes
)

## Simulator

In [178]:
hf_simulator = MoleculeSimulator(
    molecule_fun=hf,
    basis="cc-pVTZ",
    coord_scale=0.1,
    verbose=0,
)

### Reference molecule

In [179]:
include_kwargs = {
    "include_integrals": True,
    "include_hartree_fock": True,
    "include_cc": False,
    "include_coordinates": False,
    "include_all": False
}

hf_reference = hf_simulator.simulate(molecule_kwargs={"bond_distance": 1.75, "perturb": False}, **include_kwargs)

In [180]:
hf_reference["positions"]

array([[0.  , 0.  , 0.  ],
       [1.75, 0.  , 0.  ]], dtype=float32)

In [181]:
reference_determinant = hf_reference["determinant"]
reference_determinant.shape

(44, 44)

In [183]:
reference_overlap = hf_reference["overlaps"]
reference_overlap.shape

(44, 44)

### Sample and target molecules

In [184]:
sample_molecules = hf_simulator.sample(10, include_kwargs={"include_all": False})

Generating samples: 100%|██████████| 10/10 [00:00<00:00, 853.42it/s]


In [185]:
sample_procrustes_matrices = compute_procrustes_matrices(
    batched_atoms=sample_molecules["atoms"],
    batched_positions=sample_molecules["positions"],
    reference_determinant=reference_determinant,
    reference_overlap=reference_overlap,
)

Computing procrustes: 100%|██████████| 10/10 [00:07<00:00,  1.36it/s]


In [186]:
target_molecules = hf_simulator.sample(81, include_kwargs={"include_all": False})

Generating samples: 100%|██████████| 81/81 [00:00<00:00, 878.92it/s]


In [187]:
target_procrustes_matrices = compute_procrustes_matrices(
    batched_atoms=target_molecules["atoms"],
    batched_positions=target_molecules["positions"],
    reference_determinant=reference_determinant,
    reference_overlap=reference_overlap,
)

Computing procrustes: 100%|██████████| 81/81 [00:58<00:00,  1.38it/s]


In [192]:
# This replaces setup_sample().
procrustes_cc = compute_cc_with_procrustes(
    batched_atoms=sample_molecules["atoms"],
    batched_positions=sample_molecules["positions"],
    reference_determinant=reference_determinant,
    reference_overlap=reference_overlap,
)

Computing CCSD: 100%|██████████| 10/10 [00:23<00:00,  2.40s/it]


In [193]:
procrustes_cc.keys()

dict_keys(['t1', 't2', 'energies'])